# UC-05 · Check the NPSH margin

**Use case:** [UC-05](../docs/product/use-cases.md) - *compare the NPSH available
at the suction with the NPSH required by the pump.*

$$ NPSH_a = \frac{p_{s,abs} - p_v}{\rho\,g} + \frac{v_s^2}{2g} + z_s $$

`pump.npsh` provides `npsh_available`, `npsh3_from_sweep` (3 % head drop),
`NPSHrCurve` (least-squares polynomial of NPSHr vs flow, with an extrapolation flag),
`npsh_margin` and `check_npsh` against an `NPSHCriteria` rule.

> No public margin rule was found for API 610 / N-553, so the shipped rule is **not
> configured** and the verdict is `info`.  The rule used in section 5 is an
> *illustrative demo rule*, not a standard value.

**Acceptance:** `npsh_available` equals an independent hand computation; the NPSH3
sweep interpolates to the right value; the margin verdicts follow the rule.

## 1 · Setup

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import (
    NPSHCriteria,
    NPSHrCurve,
    Pressure,
    Q_,
    Water,
    check_npsh,
    npsh3_from_sweep,
    npsh_available,
    npsh_margin,
    plot_npsh,
)
from pump.constants import G

## 2 · NPSH available from the suction conditions

20 degC water, suction gauge reading +0.2 bar **gauge** (the reference is explicit
through `Pressure`), a 50 mm suction pipe and a pressure tap 1 m above the pump
reference, at the 9.8 m3/h duty.  The vapour pressure comes from `Water`.

In [ ]:
fluid = Water(Q_(20, "degC"))
q_rated = Q_(9.8, "m**3/h")
p_suction = Pressure(Q_(0.2, "bar"), "g")
suction_id = Q_(50, "mm")
z_suction = Q_(1.0, "m")

npsha = npsh_available(p_suction, fluid, q_rated, suction_id=suction_id, z_suction=z_suction)

# independent hand computation of the same equation
p_abs = p_suction.absolute()
pressure_head = ((p_abs - fluid.vapour_pressure) / (fluid.density * G)).to("m")
velocity = (q_rated / (math.pi * suction_id**2 / 4)).to("m/s")
velocity_head = (velocity**2 / (2 * G)).to("m")
manual = pressure_head + velocity_head + z_suction

print(f"p_abs = {p_abs.to('bar'):.4f~P}  p_v = {fluid.vapour_pressure.to('kPa'):.3f~P}")
print(f"pressure head {pressure_head:.3f~P} + velocity head {velocity_head:.3f~P} + z {z_suction:~P}")
print(f"NPSHa = {npsha:.3f~P}   (hand: {manual:.3f~P})")

## 3 · NPSHr curve (points typed in feet)

Illustrative NPSHr points `0.5 + 0.02 Q + 0.004 Q^2` metres (1.08 m at the rated
flow), entered in **feet** and in gpm as an engineer would read them from a data sheet.

In [ ]:
q_m3h = [3.4, 6.6, 9.8, 11.8]
npshr_m = [round(0.5 + 0.02 * q + 0.004 * q * q, 3) for q in q_m3h]

q_points = tuple(Q_(q, "m**3/h").to("gallon/minute") for q in q_m3h)
r_points = tuple(Q_(r, "m").to("ft") for r in npshr_m)
npshr_curve = NPSHrCurve(q=q_points, npshr=r_points, degree=2)

npshr_rated = npshr_curve.at_rated(q_rated)
print("points:", [f"{q:.1f~P} -> {r:.2f~P}" for q, r in zip(q_points, r_points)])
print(f"NPSHr at the rated flow = {npshr_rated:.3f~P}")
print("extrapolated at 20 m3/h:", npshr_curve.extrapolated(Q_(20, "m**3/h")))

## 4 · NPSH3 from a suction sweep

At a fixed flow the suction is throttled and the head is recorded.  NPSH3 is the NPSHa
at which the head has dropped by 3 % from the mean of the first two points
(linear interpolation between the bracketing points).

In [ ]:
sweep_npsha = [Q_(v, "m") for v in (8, 7, 6, 5, 4, 3)]
sweep_head = [Q_(v, "m") for v in (100, 100, 100, 90, 80, 70)]
npsh3 = npsh3_from_sweep(sweep_npsha, sweep_head)
print(f"NPSH3 = {npsh3:.3f~P}")

try:
    npsh3_from_sweep(sweep_npsha, [Q_(100, "m")] * 6)
    never_drops = False
except ValueError as exc:
    never_drops = True
    print("ValueError:", exc)

## 5 · Ratio, margin and the verdict

Without a configured rule the status is `info`.  With the illustrative rule
`NPSHa / NPSHr >= 1.3` and `margin >= 0.6 m` it becomes `pass` or `fail`.  A second
scenario with a vacuum suction (-0.8 bar gauge) has a much lower NPSHa.

In [ ]:
low = npsh_available(
    Pressure(Q_(-0.8, "bar"), "g"), fluid, q_rated, suction_id=suction_id, z_suction=z_suction
)
rule = NPSHCriteria(min_ratio=1.3, min_margin=Q_(0.6, "m"), source="illustrative demo rule")
strict = NPSHCriteria(min_ratio=3.5, source="illustrative strict rule")

margin = npsh_margin(npsha, npshr_rated)
print(f"ratio = {margin['ratio']:.2f}   margin = {margin['margin']:.2f~P}")

results = {
    "no rule": check_npsh(npsha, npshr_rated),
    "demo rule": check_npsh(npsha, npshr_rated, rule),
    "low suction, demo rule": check_npsh(low, npshr_rated, rule),
    "low suction, strict rule": check_npsh(low, npshr_rated, strict),
}
for name, res in results.items():
    print(f"{name:<26} NPSHa {res.npsha:6.2f~P}  ratio {res.ratio:5.2f}  -> {res.status}")

In [ ]:
png = plot_npsh(npshr_curve, npsha=npsha, rated_q=q_rated)
display(Image(data=png.getvalue()))

## 6 · Acceptance

In [ ]:
# NPSHa equals the independent computation (and has the right order of magnitude)
assert abs((npsha - manual).to("m").magnitude) < 1e-9
assert 13.0 < npsha.to("m").magnitude < 13.6
assert 2.9 < low.to("m").magnitude < 3.2

# the fitted NPSHr reproduces the rated point and is not extrapolated there
assert abs(npshr_rated.to("m").magnitude - 1.08) < 0.01
assert npshr_curve.extrapolated(q_rated) is False
assert npshr_curve.extrapolated(Q_(20, "m**3/h")) is True

# NPSH3 by interpolation (the sweep example of the docstring)
assert abs(npsh3.to("m").magnitude - 5.7) < 1e-9
assert never_drops

# margin arithmetic and verdicts
assert abs(margin["ratio"] - npsha.to("m").magnitude / npshr_rated.to("m").magnitude) < 1e-9
assert results["no rule"].status == "info"
assert results["demo rule"].status == "pass"
assert results["low suction, demo rule"].status == "pass"
assert results["low suction, strict rule"].status == "fail"
assert NPSHCriteria().configured is False
print("UC-05 acceptance OK")